# 📊 SPA-VAEGAN: Phase 4 — Benchmarking, Publication Figures & Deployment

### **Project: 4× Image Super-Resolution ($64\times 64 \to 256\times 256$)**
**Author:** Bevan (@Bevan)  
**Academic Context:** Advanced Generative AI Continuous Assessment (CAC)

---

### **Executive Summary & Scope**
This notebook executes the final evaluation, benchmarking, and deployment workflows:
1. **Held-Out Test Set Evaluation:** Evaluates Bicubic, Baseline CVAE, and SPA-VAEGAN across all **2,000 held-out test patches** (100 official DIV2K validation images).
2. **Seven Quantitative Metrics:** Computes MSE, MAE, PSNR, SSIM, LPIPS, FID, and Inference Latency.
3. **Statistical Significance Testing:** Computes 95% paired bootstrap confidence intervals (1,000 resamples) for the metric differences between SPA-VAEGAN and Baseline CVAE.
4. **Latency & Parameter Profiling:** Benchmarks inference runtime on GPU and CPU.
5. **Publication-Ready Figures:** Generates multi-panel comparison grids with zoomed crop insets and Sobel edge-map visualizations.
6. **Model Weight Export:** Strips optimizer states and discriminators, producing compact ~7 MB generator checkpoints ready for deployment.
7. **Streamlit Deployment Smoke Test:** Previews the web application.


## 1. Environment & Codebase Setup

### **Purpose of Code**
Verifies GPU state, loads codebase, and prepares directories.


In [ ]:
# ==============================================================================
# ENVIRONMENT & REPOSITORY SETUP (GOOGLE COLAB & LOCAL)
# ==============================================================================
# In Google Colab, you have TWO easy ways to load the project codebase:
# OPTION 1: Your GitHub repository URL is pre-configured below:
GITHUB_REPO_URL = "https://github.com/bevanmathewc/SPA-VAEGAN.git"

# OPTION 2: Simply drag & drop 'SPA-VAEGAN.zip' from your PC into Colab's left 
# sidebar file browser (Files icon)! It will be automatically detected and unzipped.
# ==============================================================================

import os
import sys
import torch

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IS_COLAB = True
except ImportError:
    print("[i] Running in local/offline environment (not Google Colab).")
    IS_COLAB = False

if IS_COLAB:
    REPO_DIR = '/content/drive/MyDrive/SPA-VAEGAN'
    
    # Check if codebase exists in Drive
    if not os.path.exists(os.path.join(REPO_DIR, 'src')):
        print(f"Repository source code not found in Drive at {REPO_DIR}.")
        
        # Check for uploaded zip archive
        zip_candidates = [
            '/content/SPA-VAEGAN.zip',
            '/content/GENAI.zip',
            '/content/drive/MyDrive/SPA-VAEGAN.zip',
            '/content/drive/MyDrive/GENAI.zip'
        ]
        unzipped = False
        for z in zip_candidates:
            if os.path.exists(z):
                print(f"[✓] Found uploaded zip archive: {z}")
                print(f"Extracting into {REPO_DIR}...")
                import zipfile
                os.makedirs(REPO_DIR, exist_ok=True)
                with zipfile.ZipFile(z, 'r') as zf:
                    zf.extractall(REPO_DIR)
                unzipped = True
                print("[✓] Unzipped successfully!")
                break
                
        # If not unzipped, try git clone
        if not unzipped:
            if "YOUR_USERNAME" not in GITHUB_REPO_URL:
                print(f"[✓] Cloning repository from GitHub: {GITHUB_REPO_URL}...")
                !git clone {GITHUB_REPO_URL} {REPO_DIR}
            else:
                if os.path.exists('/content/src'):
                    REPO_DIR = '/content'
                    print("[✓] Found source files in /content directly.")
                else:
                    print("\n" + "="*70)
                    print("[!] ACTION REQUIRED TO LOAD CODE IN COLAB:")
                    print("Either:")
                    print("  A) Set GITHUB_REPO_URL above to your GitHub repo URL and re-run this cell, OR")
                    print("  B) Drag & drop 'SPA-VAEGAN.zip' into Colab's left sidebar file browser and re-run this cell.")
                    print("="*70 + "\n")

    # Use %cd magic so BOTH Python and !shell commands execute from the repo root
    if os.path.exists(REPO_DIR) and os.path.exists(os.path.join(REPO_DIR, 'src')):
        %cd {REPO_DIR}
    elif os.path.exists('/content/src'):
        %cd /content

    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())
    os.environ["PYTHONPATH"] = os.getcwd()

print(f"\n[✓] Current Working Directory: {os.getcwd()}")
if os.path.exists("src"):
    print("[✓] Project codebase successfully verified (Found 'src', 'models', 'configs').")
else:
    print("[!] Warning: 'src' folder still not found. Please upload 'SPA-VAEGAN.zip' or set GITHUB_REPO_URL.")

# Hardware inspection
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM Allocated:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    !nvidia-smi
else:
    print("[!] Warning: No GPU detected. In Colab: Runtime -> Change runtime type -> T4 GPU.")


## 2. Held-Out Test Set Evaluation (`evaluation/evaluate.py`)

### **Purpose of Code**
Executes the standardized evaluation protocol across the 2,000 test patches. Generates `results/benchmark.csv` and `results/per_image_metrics.csv`.


In [ ]:
# Run test evaluation
PATCHES_DIR = "/content/patches" if os.path.exists("/content/patches") else "dataset/patches"
test_npy = os.path.join(PATCHES_DIR, "test_patches.npy")

!python -m evaluation.evaluate --config configs/default.yaml --test_npy {test_npy}


## 3. Benchmark Table & Statistical Bootstrap Analysis

### **Purpose of Code**
Displays the final benchmark table and interprets the 95% bootstrap confidence intervals for the differences between SPA-VAEGAN and the Baseline CVAE.


In [ ]:
import pandas as pd

bench_csv = "results/benchmark.csv"
if os.path.exists(bench_csv):
    df = pd.read_csv(bench_csv)
    display(df)
else:
    print("[!] Benchmark CSV not found.")


### 🔍 **Interpretation: Perception–Distortion Trade-off Analysis**
- **The Perception–Distortion Trade-off (Blau & Michaeli, 2018):** Mathematically, models trained solely with distortion metrics (L1/MSE) achieve higher PSNR and SSIM by predicting the conditional mean image. However, conditional mean images appear blurry to human observers.
- **Perceptual Improvement:** SPA-VAEGAN trades a modest decrease in PSNR/SSIM for substantial, statistically significant reductions in **LPIPS** and **FID**, producing sharp textures and natural edges.
- **Bootstrap Confidence Intervals:** If the 95% bootstrap confidence interval for $\Delta\text{LPIPS}$ strictly excludes zero, the perceptual sharpness improvement is statistically significant and not an artifact of test sample selection.


## 4. Latency & Parameter Profiling (`evaluation/latency.py`)

### **Purpose of Code**
Measures inference latency (batch size 1, $64\times 64 \to 256\times 256$) over 100 timed trials with CUDA synchronization, and counts model parameters.


In [ ]:
!python -m evaluation.latency --config configs/default.yaml


## 5. Visual Comparison Grids & Edge Maps (`evaluation/make_figures.py`)

### **Purpose of Code**
Generates multi-column comparison grids with zoomed crop insets and Sobel edge-map visualizations.


In [ ]:
!python -m evaluation.make_figures --test_npy {test_npy} --out_dir results/figures

from IPython.display import Image, display

if os.path.exists("results/figures/comparison_grid.png"):
    print("Visual Comparison Grid:")
    display(Image("results/figures/comparison_grid.png", width=900))

if os.path.exists("results/figures/edge_maps.png"):
    print("\nSobel Edge Map Comparison:")
    display(Image("results/figures/edge_maps.png", width=900))


## 6. Checkpoint Stripping & Generator Export (`src/export_model.py`)

### **Purpose of Code**
Full training checkpoints exceed 250 MB because they contain Adam optimizer momentums and discriminator parameters. For deployment, we export lightweight generator state dictionaries (~7 MB), well within GitHub's 100 MB file limit.


In [ ]:
# Export Baseline Generator
!python -m src.export_model --input checkpoints/baseline/best.pt --output checkpoints/cvae_baseline_G.pt

# Export SPA-VAEGAN Generator
!python -m src.export_model --input checkpoints/spa_vaegan/best.pt --output checkpoints/spa_vaegan_G.pt

print("\n--- Exported Checkpoint Sizes ---")
for fname in ["cvae_baseline_G.pt", "spa_vaegan_G.pt"]:
    p = os.path.join("checkpoints", fname)
    if os.path.exists(p):
        mb = os.path.getsize(p) / (1024 * 1024)
        print(f"{fname:22s}: {mb:.2f} MB")


## 7. Streamlit Web Application Smoke Test

### **Purpose of Code**
Tests the inference engine and tiled upscaling pipeline on a sample image.


In [ ]:
import numpy as np
from PIL import Image
from app.inference import load_model, upscale_tiled

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = load_model("checkpoints/spa_vaegan_G.pt", device=str(dev))

# Smoke test on synthetic sample image
sample_img = Image.open("app/assets/sample.png").convert("RGB")
img_np = np.array(sample_img)

sr_np, latency_ms = upscale_tiled(model, img_np, device=dev, tile=64, halo=24)
print(f"[✓] Tiled inference successful! Output shape: {sr_np.shape}, Elapsed: {latency_ms:.1f} ms")

sr_img = Image.fromarray(sr_np)
display(sr_img)


### 🔍 **Deployment Notes & Summary**
To run the full interactive web application locally:
```bash
streamlit run app/app.py
```
- **Supported Inputs:** JPG, JPEG, PNG (up to 10 MB).
- **Arbitrary Image Dimensions:** Processed seamlessly via overlap-and-crop tiling with a 24-pixel halo to eliminate border artifacts.
- **Ready for GitHub:** All code, notebooks, exported weights, documentation, and split lists are committed and structured cleanly.
